In [1]:
import pyspark

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/19 15:42:34 WARN Utils: Your hostname, MarkABaltazar, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/06/19 15:42:34 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/19 15:42:35 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
df_green = spark.read.parquet('../data/pq/green/*/*')

26/06/19 15:43:10 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: ../data/pq/green/*/*.
java.io.FileNotFoundException: File ../data/pq/green/*/* does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:384)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:143)
	at org.apache.spark.sql.catalyst.analysis.Resol

```
SELECT 
    -- Revenue grouping 
    date_trunc('hour', lpep_pickup_datetime) AS hour,
    PULocationID AS revenue_zone,

    -- Revenue calculation 
    SUM(total_amount) AS revenue_monthly_total_amount,
    COUNT(1) AS member_records
FROM
    green_data
WHERE lpep_pickup_datetime >= '2020-01-01 00:00:00'
GROUP BY
    hour, revenue_zone
ORDER BY
    hour, revenue_zone
```

In [5]:
df_green.rdd

MapPartitionsRDD[7] at javaToPython at NativeMethodAccessorImpl.java:0

In [6]:
df_green.rdd.take(5)

[Row(VendorID=2, lpep_pickup_datetime=datetime.datetime(2020, 1, 14, 13, 28, 9), lpep_dropoff_datetime=datetime.datetime(2020, 1, 14, 13, 35, 19), store_and_fwd_flag='N', RatecodeID=1, PULocationID=74, DOLocationID=75, passenger_count=1, trip_distance=1.35, fare_amount=7.0, extra=0.0, mta_tax=0.5, tip_amount=1.56, tolls_amount=0.0, ehail_fee=None, improvement_surcharge=0.3, total_amount=9.36, payment_type=1, trip_type=1, congestion_surcharge=0.0),
 Row(VendorID=2, lpep_pickup_datetime=datetime.datetime(2020, 1, 28, 12, 11, 44), lpep_dropoff_datetime=datetime.datetime(2020, 1, 28, 12, 25, 21), store_and_fwd_flag='N', RatecodeID=1, PULocationID=129, DOLocationID=179, passenger_count=1, trip_distance=3.73, fare_amount=13.5, extra=0.0, mta_tax=0.5, tip_amount=1.0, tolls_amount=0.0, ehail_fee=None, improvement_surcharge=0.3, total_amount=15.3, payment_type=1, trip_type=1, congestion_surcharge=0.0),
 Row(VendorID=None, lpep_pickup_datetime=datetime.datetime(2020, 1, 4, 11, 38), lpep_dropoff_

```
SELECT 
date_trunc('hour', lpep_pickup_datetime) AS hour,
PULocationID AS revenue_zone, 
SUM(total_amount) AS revenue_monthly_total_amount
```

In [7]:
rdd = df_green \
    .select('lpep_pickup_datetime', 'PULocationID', 'total_amount') \
    .rdd

In [8]:
rdd.take(5)

[Row(lpep_pickup_datetime=datetime.datetime(2020, 1, 14, 13, 28, 9), PULocationID=74, total_amount=9.36),
 Row(lpep_pickup_datetime=datetime.datetime(2020, 1, 28, 12, 11, 44), PULocationID=129, total_amount=15.3),
 Row(lpep_pickup_datetime=datetime.datetime(2020, 1, 4, 11, 38), PULocationID=25, total_amount=28.88),
 Row(lpep_pickup_datetime=datetime.datetime(2020, 1, 30, 10, 3, 36), PULocationID=182, total_amount=12.3),
 Row(lpep_pickup_datetime=datetime.datetime(2020, 1, 30, 20, 56, 17), PULocationID=210, total_amount=7.0)]

```
WHERE lpep_pickup_datetime >= '2020-01-01 00:00:00'
```

In [9]:
rdd.filter(lambda row: True).take(1)

[Row(lpep_pickup_datetime=datetime.datetime(2020, 1, 14, 13, 28, 9), PULocationID=74, total_amount=9.36)]

In [10]:
rdd.filter(lambda row: False).take(1)

[]

In [11]:
from datetime import datetime

In [12]:
start = datetime(year=2020, month=1, day=1)

In [13]:
rdd.filter(lambda row: row.lpep_pickup_datetime >= start).take(1)

[Row(lpep_pickup_datetime=datetime.datetime(2020, 1, 14, 13, 28, 9), PULocationID=74, total_amount=9.36)]

In [14]:
start = datetime(year=2020, month=1, day=1)

def filter_outliers(row):
    return row.lpep_pickup_datetime >= start

In [15]:
rdd.filter(filter_outliers).take(1)

[Row(lpep_pickup_datetime=datetime.datetime(2020, 1, 14, 13, 28, 9), PULocationID=74, total_amount=9.36)]

```
SELECT
    date_trunc('hour', lpep_pickup_datetime) AS hour,
    SUM(total_amount) AS revenue_monthly_total_amount,
    COUNT(1) AS member_records
GROUP BY
    hour, revenue_zone
```

In [17]:
rows = rdd.take(10)
row = rows[0]

row

Row(lpep_pickup_datetime=datetime.datetime(2020, 1, 14, 13, 28, 9), PULocationID=74, total_amount=9.36)

In [19]:
row.lpep_pickup_datetime.replace(minute=0, second=0, microsecond=0)

datetime.datetime(2020, 1, 14, 13, 0)

In [20]:
def prepare_for_grouping(row):
    hour = row.lpep_pickup_datetime.replace(minute=0, second=0, microsecond=0)
    zone = row.PULocationID
    key = (hour, zone)

    amount = row.total_amount
    count = 1
    value = (amount, count)

    return (key, value)

In [21]:
rdd \
    .filter(filter_outliers) \
    .map(prepare_for_grouping) \
    .take(5)

[((datetime.datetime(2020, 1, 14, 13, 0), 74), (9.36, 1)),
 ((datetime.datetime(2020, 1, 28, 12, 0), 129), (15.3, 1)),
 ((datetime.datetime(2020, 1, 4, 11, 0), 25), (28.88, 1)),
 ((datetime.datetime(2020, 1, 30, 10, 0), 182), (12.3, 1)),
 ((datetime.datetime(2020, 1, 30, 20, 0), 210), (7.0, 1))]

In [23]:
def calculate_revenue(left_value, right_value):
    left_amount, left_count = left_value
    right_amount, right_count = right_value

    output_amount = left_amount + right_amount
    output_count = left_count + right_count

    return (output_amount, output_count)

In [25]:
rdd \
    .filter(filter_outliers) \
    .map(prepare_for_grouping) \
    .reduceByKey(calculate_revenue) \
    .take(10)

[((datetime.datetime(2020, 1, 14, 13, 0), 74), (717.9299999999997, 56)),
 ((datetime.datetime(2020, 1, 30, 16, 0), 33), (653.6500000000001, 37)),
 ((datetime.datetime(2020, 1, 14, 8, 0), 166), (437.17000000000013, 35)),
 ((datetime.datetime(2020, 1, 14, 18, 0), 75), (1333.4099999999992, 87)),
 ((datetime.datetime(2020, 1, 17, 13, 0), 244), (614.63, 23)),
 ((datetime.datetime(2020, 1, 28, 6, 0), 128), (50.34, 1)),
 ((datetime.datetime(2020, 1, 2, 11, 0), 52), (252.85000000000005, 16)),
 ((datetime.datetime(2020, 1, 15, 17, 0), 7), (417.21000000000026, 37)),
 ((datetime.datetime(2020, 1, 9, 14, 0), 7), (306.61000000000007, 26)),
 ((datetime.datetime(2020, 1, 17, 22, 0), 254), (61.43000000000001, 2))]

In [26]:
def unwrap(row):
    return (row[0][0], row[0][1], row[1][0], row[1][1])

In [27]:
rdd \
    .filter(filter_outliers) \
    .map(prepare_for_grouping) \
    .reduceByKey(calculate_revenue) \
    .map(unwrap) \
    .take(10)

[(datetime.datetime(2020, 1, 14, 13, 0), 74, 717.9299999999997, 56),
 (datetime.datetime(2020, 1, 30, 16, 0), 33, 653.6500000000001, 37),
 (datetime.datetime(2020, 1, 14, 8, 0), 166, 437.17000000000013, 35),
 (datetime.datetime(2020, 1, 14, 18, 0), 75, 1333.4099999999992, 87),
 (datetime.datetime(2020, 1, 17, 13, 0), 244, 614.63, 23),
 (datetime.datetime(2020, 1, 28, 6, 0), 128, 50.34, 1),
 (datetime.datetime(2020, 1, 2, 11, 0), 52, 252.85000000000005, 16),
 (datetime.datetime(2020, 1, 15, 17, 0), 7, 417.21000000000026, 37),
 (datetime.datetime(2020, 1, 9, 14, 0), 7, 306.61000000000007, 26),
 (datetime.datetime(2020, 1, 17, 22, 0), 254, 61.43000000000001, 2)]

In [28]:
rdd \
    .filter(filter_outliers) \
    .map(prepare_for_grouping) \
    .reduceByKey(calculate_revenue) \
    .map(unwrap) \
    .toDF() \
    .show()

+-------------------+---+------------------+---+
|                 _1| _2|                _3| _4|
+-------------------+---+------------------+---+
|2020-01-14 13:00:00| 74| 717.9299999999997| 56|
|2020-01-30 16:00:00| 33| 653.6500000000001| 37|
|2020-01-14 08:00:00|166|437.17000000000013| 35|
|2020-01-14 18:00:00| 75|1333.4099999999992| 87|
|2020-01-17 13:00:00|244|            614.63| 23|
|2020-01-28 06:00:00|128|             50.34|  1|
|2020-01-02 11:00:00| 52|252.85000000000005| 16|
|2020-01-15 17:00:00|  7|417.21000000000026| 37|
|2020-01-09 14:00:00|  7|306.61000000000007| 26|
|2020-01-17 22:00:00|254| 61.43000000000001|  2|
|2020-01-13 09:00:00|225|            317.38| 12|
|2020-01-24 15:00:00|181|180.89000000000001| 13|
|2020-01-18 16:00:00| 22|              37.3|  1|
|2020-01-07 08:00:00| 74| 1497.259999999998|104|
|2020-01-05 08:00:00| 42| 338.2700000000001| 18|
|2020-01-28 20:00:00| 66|304.14000000000004| 16|
|2020-01-19 14:00:00|166|396.83000000000004| 28|
|2020-01-04 21:00:00

Traceback (most recent call last):
  File "/home/markanthony/projects/DE_Zoomcamp/06-batch-processing/.venv/lib/python3.12/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 233, in manager
    code = worker(sock, authenticated)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/markanthony/projects/DE_Zoomcamp/06-batch-processing/.venv/lib/python3.12/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 87, in worker
    outfile.flush()
BrokenPipeError: [Errno 32] Broken pipe


In [29]:
from collections import namedtuple

In [30]:
RevenueRow = namedtuple('RevenueRow', ['hour','zone','revenue','count'])

In [31]:
def unwrap(row):
    return RevenueRow(
        hour=row[0][0], 
        zone=row[0][1], 
        revenue=row[1][0], 
        count=row[1][1]
    )

In [32]:
rdd \
    .filter(filter_outliers) \
    .map(prepare_for_grouping) \
    .reduceByKey(calculate_revenue) \
    .map(unwrap) \
    .toDF() \
    .show()

+-------------------+----+------------------+-----+
|               hour|zone|           revenue|count|
+-------------------+----+------------------+-----+
|2020-01-14 13:00:00|  74| 717.9299999999997|   56|
|2020-01-30 16:00:00|  33| 653.6500000000001|   37|
|2020-01-14 08:00:00| 166|437.17000000000013|   35|
|2020-01-14 18:00:00|  75|1333.4099999999992|   87|
|2020-01-17 13:00:00| 244|            614.63|   23|
|2020-01-28 06:00:00| 128|             50.34|    1|
|2020-01-02 11:00:00|  52|252.85000000000005|   16|
|2020-01-15 17:00:00|   7|417.21000000000026|   37|
|2020-01-09 14:00:00|   7|306.61000000000007|   26|
|2020-01-17 22:00:00| 254| 61.43000000000001|    2|
|2020-01-13 09:00:00| 225|            317.38|   12|
|2020-01-24 15:00:00| 181|180.89000000000001|   13|
|2020-01-18 16:00:00|  22|              37.3|    1|
|2020-01-07 08:00:00|  74| 1497.259999999998|  104|
|2020-01-05 08:00:00|  42| 338.2700000000001|   18|
|2020-01-28 20:00:00|  66|304.14000000000004|   16|
|2020-01-19 

Traceback (most recent call last):
  File "/home/markanthony/projects/DE_Zoomcamp/06-batch-processing/.venv/lib/python3.12/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 233, in manager
    code = worker(sock, authenticated)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/markanthony/projects/DE_Zoomcamp/06-batch-processing/.venv/lib/python3.12/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 87, in worker
    outfile.flush()
BrokenPipeError: [Errno 32] Broken pipe


In [33]:
df_result = rdd \
    .filter(filter_outliers) \
    .map(prepare_for_grouping) \
    .reduceByKey(calculate_revenue) \
    .map(unwrap) \
    .toDF()

In [34]:
df_result.schema

StructType([StructField('hour', TimestampType(), True), StructField('zone', LongType(), True), StructField('revenue', DoubleType(), True), StructField('count', LongType(), True)])

In [35]:
from pyspark.sql import types

In [36]:
result_schema = types.StructType([
    types.StructField('hour', types.TimestampType(), True), 
    types.StructField('zone', types.IntegerType(), True), 
    types.StructField('revenue', types.DoubleType(), True), 
    types.StructField('count', types.IntegerType(), True)
])

In [37]:
df_result = rdd \
    .filter(filter_outliers) \
    .map(prepare_for_grouping) \
    .reduceByKey(calculate_revenue) \
    .map(unwrap) \
    .toDF(result_schema)

In [38]:
df_result.show()

[Stage 28:==================================================>       (7 + 1) / 8]

+-------------------+----+------------------+-----+
|               hour|zone|           revenue|count|
+-------------------+----+------------------+-----+
|2020-01-14 13:00:00|  74| 717.9299999999997|   56|
|2020-01-30 16:00:00|  33| 653.6500000000001|   37|
|2020-01-14 08:00:00| 166|437.17000000000013|   35|
|2020-01-14 18:00:00|  75|1333.4099999999992|   87|
|2020-01-17 13:00:00| 244|            614.63|   23|
|2020-01-28 06:00:00| 128|             50.34|    1|
|2020-01-02 11:00:00|  52|252.85000000000005|   16|
|2020-01-15 17:00:00|   7|417.21000000000026|   37|
|2020-01-09 14:00:00|   7|306.61000000000007|   26|
|2020-01-17 22:00:00| 254| 61.43000000000001|    2|
|2020-01-13 09:00:00| 225|            317.38|   12|
|2020-01-24 15:00:00| 181|180.89000000000001|   13|
|2020-01-18 16:00:00|  22|              37.3|    1|
|2020-01-07 08:00:00|  74| 1497.259999999998|  104|
|2020-01-05 08:00:00|  42| 338.2700000000001|   18|
|2020-01-28 20:00:00|  66|304.14000000000004|   16|
|2020-01-19 

Traceback (most recent call last):                                              
  File "/home/markanthony/projects/DE_Zoomcamp/06-batch-processing/.venv/lib/python3.12/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 233, in manager
    code = worker(sock, authenticated)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/markanthony/projects/DE_Zoomcamp/06-batch-processing/.venv/lib/python3.12/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 87, in worker
    outfile.flush()
BrokenPipeError: [Errno 32] Broken pipe


In [39]:
df_result.write.parquet('../tmp/green-revenue')

26/06/19 16:48:44 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
                                                                                

In [40]:
df_green.show()

+--------+--------------------+---------------------+------------------+----------+------------+------------+---------------+-------------+-----------+-----+-------+----------+------------+---------+---------------------+------------+------------+---------+--------------------+
|VendorID|lpep_pickup_datetime|lpep_dropoff_datetime|store_and_fwd_flag|RatecodeID|PULocationID|DOLocationID|passenger_count|trip_distance|fare_amount|extra|mta_tax|tip_amount|tolls_amount|ehail_fee|improvement_surcharge|total_amount|payment_type|trip_type|congestion_surcharge|
+--------+--------------------+---------------------+------------------+----------+------------+------------+---------------+-------------+-----------+-----+-------+----------+------------+---------+---------------------+------------+------------+---------+--------------------+
|       2| 2020-01-14 13:28:09|  2020-01-14 13:35:19|                 N|         1|          74|          75|              1|         1.35|        7.0|  0.0|    0.

In [41]:
columns = ['VendorID', 'lpep_pickup_datetime', 'PULocationID', 'DOLocationID', 'trip_distance']

df_green \
    .select(columns) \
    .show()

+--------+--------------------+------------+------------+-------------+
|VendorID|lpep_pickup_datetime|PULocationID|DOLocationID|trip_distance|
+--------+--------------------+------------+------------+-------------+
|       2| 2020-01-14 13:28:09|          74|          75|         1.35|
|       2| 2020-01-28 12:11:44|         129|         179|         3.73|
|    NULL| 2020-01-04 11:38:00|          25|          41|        11.19|
|       2| 2020-01-30 10:03:36|         182|         242|         1.32|
|       1| 2020-01-30 20:56:17|         210|         210|          0.8|
|       2| 2020-01-06 14:40:15|          33|          66|         1.24|
|       2| 2020-01-09 18:01:49|          42|         116|         0.52|
|       2| 2020-01-29 04:39:21|         116|         116|         0.58|
|    NULL| 2020-01-09 15:01:00|         223|          92|         6.79|
|       2| 2020-01-31 08:32:25|          55|          55|         9.69|
|       2| 2020-01-15 19:24:18|           7|         179|       

In [42]:
duration_rdd = df_green \
    .select(columns) \
    .rdd

In [43]:
duration_rdd.take(5)

[Row(VendorID=2, lpep_pickup_datetime=datetime.datetime(2020, 1, 14, 13, 28, 9), PULocationID=74, DOLocationID=75, trip_distance=1.35),
 Row(VendorID=2, lpep_pickup_datetime=datetime.datetime(2020, 1, 28, 12, 11, 44), PULocationID=129, DOLocationID=179, trip_distance=3.73),
 Row(VendorID=None, lpep_pickup_datetime=datetime.datetime(2020, 1, 4, 11, 38), PULocationID=25, DOLocationID=41, trip_distance=11.19),
 Row(VendorID=2, lpep_pickup_datetime=datetime.datetime(2020, 1, 30, 10, 3, 36), PULocationID=182, DOLocationID=242, trip_distance=1.32),
 Row(VendorID=1, lpep_pickup_datetime=datetime.datetime(2020, 1, 30, 20, 56, 17), PULocationID=210, DOLocationID=210, trip_distance=0.8)]

In [46]:
def apply_model_in_batch(partition):
    cnt = 0

    for row in partition:
        cnt = cnt + 1
        
    return [cnt]

In [47]:
rdd.mapPartitions(apply_model_in_batch).collect()

[746744, 418184, 219219, 215981, 212334, 199145, 183795, 109115]

In [48]:
import pandas as pd

In [49]:
rows = duration_rdd.take(10)

In [50]:
pd.DataFrame(rows, columns=columns)

,VendorID,lpep_pickup_datetime,PULocationID,DOLocationID,trip_distance
0,2.0,2020-01-14 13:28:09,74,75,1.35
1,2.0,2020-01-28 12:11:44,129,179,3.73
2,NaN,2020-01-04 11:38:00,25,41,11.19
3,2.0,2020-01-30 10:03:36,182,242,1.32
4,1.0,2020-01-30 20:56:17,210,210,0.80
5,2.0,2020-01-06 14:40:15,33,66,1.24
6,2.0,2020-01-09 18:01:49,42,116,0.52
7,2.0,2020-01-29 04:39:21,116,116,0.58
8,NaN,2020-01-09 15:01:00,223,92,6.79
9,2.0,2020-01-31 08:32:25,55,55,9.69


In [51]:
def apply_model_in_batch(rows):
    df = pd.DataFrame(rows, columns=columns)
    cnt = len(df)
    return [cnt]

In [53]:
duration_rdd.mapPartitions(apply_model_in_batch).collect()

[746744, 418184, 219219, 215981, 212334, 199145, 183795, 109115]

In [55]:
# model = ...

def model_predict(df):
    # y_pred = model.predict(df)
    y_pred = df.trip_distance * 5
    return y_pred

In [56]:
def apply_model_in_batch(rows):
    df = pd.DataFrame(rows, columns=columns)
    predictions = model_predict(df)
    df['predicted_duration'] = predictions

    for row in df.itertuples():
        yield row

In [57]:
duration_rdd.mapPartitions(apply_model_in_batch).take(10)

[Pandas(Index=0, VendorID=2.0, lpep_pickup_datetime=Timestamp('2020-01-14 13:28:09'), PULocationID=74, DOLocationID=75, trip_distance=1.35, predicted_duration=6.75),
 Pandas(Index=1, VendorID=2.0, lpep_pickup_datetime=Timestamp('2020-01-28 12:11:44'), PULocationID=129, DOLocationID=179, trip_distance=3.73, predicted_duration=18.65),
 Pandas(Index=2, VendorID=nan, lpep_pickup_datetime=Timestamp('2020-01-04 11:38:00'), PULocationID=25, DOLocationID=41, trip_distance=11.19, predicted_duration=55.949999999999996),
 Pandas(Index=3, VendorID=2.0, lpep_pickup_datetime=Timestamp('2020-01-30 10:03:36'), PULocationID=182, DOLocationID=242, trip_distance=1.32, predicted_duration=6.6000000000000005),
 Pandas(Index=4, VendorID=1.0, lpep_pickup_datetime=Timestamp('2020-01-30 20:56:17'), PULocationID=210, DOLocationID=210, trip_distance=0.8, predicted_duration=4.0),
 Pandas(Index=5, VendorID=2.0, lpep_pickup_datetime=Timestamp('2020-01-06 14:40:15'), PULocationID=33, DOLocationID=66, trip_distance=1.

In [58]:
duration_rdd.mapPartitions(apply_model_in_batch).toDF().show()

Traceback (most recent call last):                                              
  File "/home/markanthony/projects/DE_Zoomcamp/06-batch-processing/.venv/lib/python3.12/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 233, in manager
    code = worker(sock, authenticated)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/markanthony/projects/DE_Zoomcamp/06-batch-processing/.venv/lib/python3.12/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 87, in worker
    outfile.flush()
BrokenPipeError: [Errno 32] Broken pipe


+-----+--------+--------------------+------------+------------+-------------+------------------+
|Index|VendorID|lpep_pickup_datetime|PULocationID|DOLocationID|trip_distance|predicted_duration|
+-----+--------+--------------------+------------+------------+-------------+------------------+
|    0|     2.0|                  {}|          74|          75|         1.35|              6.75|
|    1|     2.0|                  {}|         129|         179|         3.73|             18.65|
|    2|     NaN|                  {}|          25|          41|        11.19|55.949999999999996|
|    3|     2.0|                  {}|         182|         242|         1.32|6.6000000000000005|
|    4|     1.0|                  {}|         210|         210|          0.8|               4.0|
|    5|     2.0|                  {}|          33|          66|         1.24|               6.2|
|    6|     2.0|                  {}|          42|         116|         0.52|               2.6|
|    7|     2.0|              

In [59]:
df_predicts = duration_rdd \
    .mapPartitions(apply_model_in_batch) \
    .toDF() \
    .drop('Index')

In [60]:
df_predicts.show()

[Stage 44:>                                                         (0 + 1) / 1]

+--------+--------------------+------------+------------+-------------+------------------+
|VendorID|lpep_pickup_datetime|PULocationID|DOLocationID|trip_distance|predicted_duration|
+--------+--------------------+------------+------------+-------------+------------------+
|     2.0|                  {}|          74|          75|         1.35|              6.75|
|     2.0|                  {}|         129|         179|         3.73|             18.65|
|     NaN|                  {}|          25|          41|        11.19|55.949999999999996|
|     2.0|                  {}|         182|         242|         1.32|6.6000000000000005|
|     1.0|                  {}|         210|         210|          0.8|               4.0|
|     2.0|                  {}|          33|          66|         1.24|               6.2|
|     2.0|                  {}|          42|         116|         0.52|               2.6|
|     2.0|                  {}|         116|         116|         0.58|               2.9|

Traceback (most recent call last):                                              
  File "/home/markanthony/projects/DE_Zoomcamp/06-batch-processing/.venv/lib/python3.12/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 233, in manager
    code = worker(sock, authenticated)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/markanthony/projects/DE_Zoomcamp/06-batch-processing/.venv/lib/python3.12/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 87, in worker
    outfile.flush()
BrokenPipeError: [Errno 32] Broken pipe


In [61]:
df_predicts.select('predicted_duration').show()

[Stage 45:>                                                         (0 + 1) / 1]

+------------------+
|predicted_duration|
+------------------+
|              6.75|
|             18.65|
|55.949999999999996|
|6.6000000000000005|
|               4.0|
|               6.2|
|               2.6|
|               2.9|
|             33.95|
|48.449999999999996|
|              6.25|
|              6.15|
|19.950000000000003|
|2.8499999999999996|
|               7.5|
|              0.95|
|              9.05|
|              7.45|
|              5.25|
|             17.35|
+------------------+
only showing top 20 rows


Traceback (most recent call last):                                              
  File "/home/markanthony/projects/DE_Zoomcamp/06-batch-processing/.venv/lib/python3.12/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 233, in manager
    code = worker(sock, authenticated)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/markanthony/projects/DE_Zoomcamp/06-batch-processing/.venv/lib/python3.12/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 87, in worker
    outfile.flush()
BrokenPipeError: [Errno 32] Broken pipe


In [62]:
spark.stop()